In [1]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecMonitor

### Add mbt-gym to path

In [2]:
import sys
sys.path.append("../../")

In [3]:
from mbt_gym.agents.BaselineAgents import CarteaJaimungalMmAgent
from mbt_gym.gym.helpers.generate_trajectory import generate_trajectory
from mbt_gym.gym.StableBaselinesTradingEnvironment import StableBaselinesTradingEnvironment
from mbt_gym.gym.TradingEnvironment import TradingEnvironment
from mbt_gym.gym.wrappers import *
from mbt_gym.rewards.RewardFunctions import PnL, CjMmCriterion
from mbt_gym.stochastic_processes.midprice_models import BrownianMotionMidpriceModel
from mbt_gym.stochastic_processes.arrival_models import PoissonArrivalModel
from mbt_gym.stochastic_processes.fill_probability_models import ExponentialFillFunction
from mbt_gym.gym.ModelDynamics import LimitOrderModelDynamics

### Create market making environment

In [4]:
import sys
sys.path.append("../") # This version of the notebook is in the subfolder "notebooks" of the repo

import gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.integrate import quad

from copy import deepcopy


from mbt_gym.agents.BaselineAgents import *
from mbt_gym.gym.TradingEnvironment import TradingEnvironment
from mbt_gym.gym.helpers.generate_trajectory import generate_trajectory
from mbt_gym.gym.helpers.plotting import *
from mbt_gym.stochastic_processes.midprice_models import *
from mbt_gym.stochastic_processes.arrival_models import *
from mbt_gym.stochastic_processes.fill_probability_models import *
import torch
#print(torch.cuda.is_available())
#print(torch.cuda.get_device_name())
from mbt_gym.gym.ModelDynamics import LimitOrderModelDynamics
seed = 42


### Function for building the enviroment

In [5]:
def get_as_env(num_trajectories:int = 1, fads_proportion:float=0.6, 
               psi:float = 15.0, phi:float = 15.0, eta:float = 10.0,
               gamma:float = 1.0,
               ) -> TradingEnvironment:
    midprice_model = ArithmeticBrownianMotionWithFadsMidpriceModelPartialInformation(initial_price=initial_price, drift=mu,
                                                 volatility=sigma, fads_proportion=fads_proportion, eta=eta, step_size=terminal_time/n_steps,
                                                 terminal_time=terminal_time,
                                                 num_trajectories=num_trajectories, seed=seed)
    arrival_model = ModifiedPoissonArrivalModel(phi=phi,
                                                psi=psi,
                                                gamma=gamma,
                                                fads_proportion=fads_proportion,
                                                sigma=sigma,
                                                step_size=terminal_time/n_steps,
                                                num_trajectories=num_trajectories,
                                                seed=seed)
    fill_probability_model = ExponentialFillFunction(fill_exponent=k, 
                                                     step_size=terminal_time/n_steps,
                                                     num_trajectories=num_trajectories,
                                                     seed=seed)
    LOtrader = LimitOrderModelDynamics(midprice_model = midprice_model, arrival_model = arrival_model, 
                                fill_probability_model = fill_probability_model,
                                num_trajectories = num_trajectories, seed=seed)
    reward = CjMmCriterion(per_step_inventory_aversion = big_phi,
                           terminal_inventory_aversion = alpha,
                           terminal_time = terminal_time)
    env_params = dict(terminal_time=terminal_time, 
                      n_steps=n_steps,
                      seed = seed,
                      initial_inventory = initial_inventory,
                      model_dynamics = LOtrader,
                      reward_function = reward,
                      max_inventory=max_inventory,
                      normalise_action_space = False,
                      normalise_observation_space = False,
                      num_trajectories=num_trajectories)
    return TradingEnvironment(**env_params)

## Varying fad proportion (paramter q)

### Parameters

In [6]:
result_list=[]

In [7]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 1000
initial_inventory = 0
fads_proportion_values = [0.0, 0.2, 0.4, 0.6, 0.8, 1]
# p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
eta = 10.0
phi = 15
k = 1
gamma = 1
alpha=0.001
mu=0
big_phi=0.1

max_inventory=20

In [8]:
# OU parameters
u0 = 0.0
xi = 1.0

def m(t):
    return u0 * np.exp(-eta * t)

def v(t):
    return (xi**2) / (2.0 * eta) * (1.0 - np.exp(-2.0 * eta * t))

def compute_psi(q):
    c = gamma * sigma * q
    def integrand(t):
        return np.exp(c * m(t) + 0.5 * (c**2) * v(t))
    denom, _ = quad(integrand, 0.0, terminal_time)
    return (30.0 - phi * terminal_time) / denom

# compute psi for each fad proportion
psi_values = {q: compute_psi(q) for q in fads_proportion_values}

print("psi values:", psi_values)

psi values: {0.0: 15.0, 0.2: 14.985756598048642, 0.4: 14.943105475282236, 0.6: 14.872283180889763, 0.8: 14.773681637922692, 1: 14.647844682743013}


In [ ]:
results_dict = {}
for q in fads_proportion_values:
    vec_env = get_as_env(
        num_trajectories=1000,
        fads_proportion=q,
        psi=psi_values[q]
    )

    vec_as = ModifiedCarteaJaimungalMmAgent(env=vec_env)

    observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(
        vec_env=vec_env, agent=vec_as
    )

    results_dict[q] = dict(
        results=results,
        rewards=total_rewards,
        obs=observations
    )
result_list.append({'fads': results_dict})  

In [10]:
header = f"{'Fads Prop':>10} | {'Psi':>10} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))

for fads_prop, result in results_dict.items():
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    print(f"{fads_prop:10.2f} | {psi_values[fads_prop]:10.4f} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15.2f} | {std_inv:13.2f}")


 Fads Prop |        Psi |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
-----------------------------------------------------------------------------------
      0.00 |    15.0000 |      21.35 |       4.95 |           -0.08 |          3.07
      0.20 |    14.9858 |      21.35 |       4.93 |           -0.09 |          3.05
      0.40 |    14.9431 |      21.27 |       4.89 |           -0.07 |          3.10
      0.60 |    14.8723 |      21.13 |       4.81 |           -0.07 |          3.11
      0.80 |    14.7737 |      20.97 |       4.66 |           -0.07 |          3.13
      1.00 |    14.6478 |      20.76 |       4.49 |           -0.06 |          3.16


## Varying eta

### Parameters

In [11]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 1000
initial_inventory = 0

fads_proportion = 0.6 # p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
eta_values = [2.5, 5, 7.5, 10.0, 12.5]
phi = 15
k = 1
gamma = 1
alpha=0.001
mu=0
big_phi=0.1

In [12]:

# OU parameters
u0 = 0.0
xi = 1.0

def m(t, eta):
    return u0 * np.exp(-eta * t)

def v(t, eta):
    return (xi**2) / (2.0 * eta) * (1.0 - np.exp(-2.0 * eta * t))

def compute_psi(q, eta):
    c = gamma * sigma * q
    def integrand(t):
        return np.exp(c * m(t, eta) + 0.5 * (c**2) * v(t, eta))
    denom, _ = quad(integrand, 0.0, terminal_time)
    return (30.0 - phi * terminal_time) / denom

# compute psi for each eta
psi_values = {eta: compute_psi(fads_proportion, eta) for eta in eta_values}

# pretty print
for eta, val in psi_values.items():
    print(f"eta={eta:.1f} -> psi={val:.6f}")

eta=2.5 -> psi=14.572885
eta=5.0 -> psi=14.758861
eta=7.5 -> psi=14.832907
eta=10.0 -> psi=14.872283
eta=12.5 -> psi=14.896670


In [ ]:
results_dict = {}
for eta in eta_values:
    vec_env = get_as_env(num_trajectories=1000, eta=eta, psi=psi_values[eta])

    vec_as = ModifiedCarteaJaimungalMmAgent(env=vec_env)

    observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(vec_env=vec_env, agent=vec_as)

    results_dict[eta] = dict(results=results, rewards=total_rewards, obs=observations)
result_list.append({'eta': results_dict}) 

In [14]:
header = f"{'Eta':>10} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))
for etas, result in results_dict.items():
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    print(f"{etas:10} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15} | {std_inv:13}")

       Eta |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
----------------------------------------------------------------------
       2.5 |      20.97 |       4.90 |          -0.042 | 3.293969641633025
         5 |      21.05 |       4.83 |          -0.028 | 3.1405757433948316
       7.5 |      21.08 |       4.80 |          -0.078 | 3.1269659416117728
      10.0 |      21.13 |       4.81 |          -0.071 | 3.106116385456282
      12.5 |      21.18 |       4.80 |          -0.082 | 3.1072939996080193


## Varying gamma parameter

### Parameters

In [15]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 1000
initial_inventory = 0
fads_proportion_values = 0.6# p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
eta = 10.0
phi = 15
k = 1
gamma_values = [0, 1, 2, 3]
alpha=0.001
mu=0
big_phi=0.1

In [16]:
# OU parameters
u0 = 0.0
xi = 1.0

def m(t, eta):
    return u0 * np.exp(-eta * t)

def v(t, eta):
    return (xi**2) / (2.0 * eta) * (1.0 - np.exp(-2.0 * eta * t))

def compute_psi(q, eta, gamma):
    c = gamma * sigma * q
    def integrand(t):
        return np.exp(c * m(t, eta) + 0.5 * (c**2) * v(t, eta))
    denom, _ = quad(integrand, 0.0, terminal_time)
    return (30.0 - phi * terminal_time) / denom

# compute psi for each gamma
psi_values = {gamma: compute_psi(fads_proportion, eta, gamma) for gamma in gamma_values}

# pretty print
for gamma, val in psi_values.items():
    print(f"gamma={gamma} -> psi={val:.6f}")

gamma=0 -> psi=15.000000
gamma=1 -> psi=14.872283
gamma=2 -> psi=14.495463
gamma=3 -> psi=13.888033


In [17]:
results_dict = {}
for gamma in gamma_values:
    vec_env = get_as_env(num_trajectories=1000, gamma=gamma, psi=psi_values[gamma])

    vec_as = ModifiedCarteaJaimungalMmAgent(env=vec_env)

    observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(vec_env=vec_env, agent=vec_as)

    results_dict[gamma] = dict(results=results, rewards=total_rewards, obs=observations)
result_list.append({'gamma': results_dict})  

In [18]:
header = f"{'Gamma':>10} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))
for gamma, result in results_dict.items():
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    print(f"{gamma:10} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15} | {std_inv:13}")

     Gamma |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
----------------------------------------------------------------------
         0 |      21.32 |       4.79 |           -0.08 | 3.070439707924583
         1 |      21.13 |       4.81 |          -0.071 | 3.106116385456282
         2 |      20.97 |       4.82 |          -0.053 | 3.211882781173684
         3 |      20.78 |       4.81 |          -0.018 | 3.3117481788324428


## Varying Informed trader proportion (psi and phi)

### Parameters

In [19]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 1000
initial_inventory = 0
fads_proportion = 0.6
# p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
eta = 10.0
k = 1
gamma = k
alpha=0.001
mu=0
big_phi=0.1

In [20]:
# OU parameters
u0 = 0.0
xi = 1.0

def m(t):
    return u0 * np.exp(-eta * t)

def v(t):
    return (xi**2) / (2.0 * eta) * (1.0 - np.exp(-2.0 * eta * t))

def compute_psi(phi):
    c = gamma * sigma * q
    def integrand(t):
        return np.exp(c * m(t) + 0.5 * (c**2) * v(t))
    denom, _ = quad(integrand, 0.0, terminal_time)
    return (30.0 - phi * terminal_time) / denom

def phi_from_informed(perc_informed):
    """
    Given the percentage of informed traders (0-100),
    compute phi according to the paper.
    """
    return 30 * (1 - perc_informed / 100.0)
def psi_from_informed(perc_informed):
    """
    Given percentage of informed traders, compute phi and psi.
    """
    phi = phi_from_informed(perc_informed)
    psi = compute_psi(phi)  # your eq (61) implementation

    return phi, psi

In [21]:
# Build the dictionary
def build_phi_psi_dict(percentages):
    results = {}
    for perc in percentages:
        phi, psi = psi_from_informed(perc)
        results[perc] = {"phi": phi, "psi": psi}
    return results

# Example usage
percentages = [0, 25, 50, 75, 100]
phi_psi_dict = build_phi_psi_dict(percentages)
for perc, vals in phi_psi_dict.items():
    print(f"{perc}% informed → phi = {vals['phi']:.2f}, psi = {vals['psi']:.4f}")



0% informed → phi = 30.00, psi = 0.0000
25% informed → phi = 22.50, psi = 7.3239
50% informed → phi = 15.00, psi = 14.6478
75% informed → phi = 7.50, psi = 21.9718
100% informed → phi = 0.00, psi = 29.2957


In [ ]:
results_dict = {}
for perc, vals in phi_psi_dict.items():
    phi, psi = vals['phi'], vals['psi']
    
    # Set up environment and agent
    vec_env = get_as_env(num_trajectories=1000, phi=phi, psi=psi)
    vec_as = ModifiedCarteaJaimungalMmAgent(env=vec_env)
    
    # Generate trajectory and results
    observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(vec_env=vec_env, agent=vec_as)
    
    # Store in results dictionary keyed by percentage
    results_dict[perc] = {
        "results": results,
        "rewards": total_rewards,
        "obs": observations
    }
result_list.append({'informed': results_dict})  

In [23]:
header = f"{'Perc':>6} | {'Phi':>8} | {'Psi':>8} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))

for perc, result in results_dict.items():
    phi, psi = phi_psi_dict[perc]['phi'], phi_psi_dict[perc]['psi']
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    
    print(f"{perc:6}% | {phi:8.2f} | {psi:8.4f} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15} | {std_inv:13}")


  Perc |      Phi |      Psi |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
----------------------------------------------------------------------------------------
     0% |    30.00 |   0.0000 |      21.32 |       4.79 |           -0.08 | 3.070439707924583
    25% |    22.50 |   7.3239 |      21.17 |       4.81 |          -0.084 | 3.0748892662988694
    50% |    15.00 |  14.6478 |      20.97 |       4.79 |          -0.067 | 3.1139221249093563
    75% |     7.50 |  21.9718 |      20.79 |       4.79 |          -0.047 | 3.142099775627757
   100% |     0.00 |  29.2957 |      20.64 |       4.81 |          -0.058 | 3.1775204169288984


### Final Summary Table

In [ ]:
fads_results = []
eta_results = []
gamma_results = []
informed_results = []

# Extract results from each experiment
for experiment_dict in result_list:
    for exp_type, results_dict in experiment_dict.items():
        for param, data in results_dict.items():
            results = data['results'].loc['Inventory']
            
            if exp_type == 'fads':
                fads_results.append({
                    'Type': 'Fads',
                    'Parameter': f'q={param:.1f}',
                    'Mean PnL': results['Mean PnL'],
                    'Std PnL': results['Std PnL'],
                    'Mean Term Inv': results['Mean terminal inventory'],
                    'Std Term Inv': results['Std terminal inventory']
                })
            elif exp_type == 'eta':
                eta_results.append({
                    'Type': 'Eta',
                    'Parameter': f'η={param:.1f}',
                    'Mean PnL': results['Mean PnL'],
                    'Std PnL': results['Std PnL'],
                    'Mean Term Inv': results['Mean terminal inventory'],
                    'Std Term Inv': results['Std terminal inventory']
                })
            elif exp_type == 'gamma':
                gamma_results.append({
                    'Type': 'Gamma',
                    'Parameter': f'γ={param}',
                    'Mean PnL': results['Mean PnL'],
                    'Std PnL': results['Std PnL'],
                    'Mean Term Inv': results['Mean terminal inventory'],
                    'Std Term Inv': results['Std terminal inventory']
                })
            elif exp_type == 'informed':
                informed_results.append({
                    'Type': 'Informed',
                    'Parameter': f'perc={param}%',
                    'Mean PnL': results['Mean PnL'],
                    'Std PnL': results['Std PnL'],
                    'Mean Term Inv': results['Mean terminal inventory'],
                    'Std Term Inv': results['Std terminal inventory']
                })

print("\nFINAL COMPLETE RESULTS SUMMARY")
print("=" * 100)

if fads_results:
    print("\nFADS PROPORTION EXPERIMENTS")
    print("-" * 100)
    df_fads = pd.DataFrame(fads_results)
    print(df_fads.to_string(index=False, float_format=lambda x: '{:.2f}'.format(x)))

if eta_results:
    print("\nETA EXPERIMENTS")
    print("-" * 100)
    df_eta = pd.DataFrame(eta_results)
    print(df_eta.to_string(index=False, float_format=lambda x: '{:.2f}'.format(x)))

if gamma_results:
    print("\nGAMMA EXPERIMENTS")
    print("-" * 100)
    df_gamma = pd.DataFrame(gamma_results)
    print(df_gamma.to_string(index=False, float_format=lambda x: '{:.2f}'.format(x)))

if informed_results:
    print("\nINFORMED TRADER PROPORTION EXPERIMENTS")
    print("-" * 100)
    df_informed = pd.DataFrame(informed_results)
    print(df_informed.to_string(index=False, float_format=lambda x: '{:.2f}'.format(x)))

print("\n" + "=" * 100)


FINAL COMPLETE RESULTS SUMMARY

FADS PROPORTION EXPERIMENTS
----------------------------------------------------------------------------------------------------
Type Parameter  Mean PnL  Std PnL  Mean Term Inv  Std Term Inv
Fads     q=0.0     21.35     4.95          -0.08          3.07
Fads     q=0.2     21.35     4.93          -0.09          3.05
Fads     q=0.4     21.27     4.89          -0.07          3.10
Fads     q=0.6     21.13     4.81          -0.07          3.11
Fads     q=0.8     20.97     4.66          -0.07          3.13
Fads     q=1.0     20.76     4.49          -0.06          3.16

ETA EXPERIMENTS
----------------------------------------------------------------------------------------------------
Type Parameter  Mean PnL  Std PnL  Mean Term Inv  Std Term Inv
 Eta     η=2.5     20.97     4.90          -0.04          3.29
 Eta     η=5.0     21.05     4.83          -0.03          3.14
 Eta     η=7.5     21.08     4.80          -0.08          3.13
 Eta    η=10.0     21.13   